# 03c — UC2 FedGen (global) with the distillation FIX

Same as 03b (global partition, `new_partitions_global/` -> `results/newpart_global/`)
but fixes the reason the previous run failed: **`ensemble_beta = 0` disabled the
student-distillation term**, so FedGen never beat FedAvg (Student Loss = 0.0000
every round; scaled MAE stuck ~0.36-0.40 vs FedAvg ~0.15).

### Why a notebook alone can't fix it — and how this one does
`ensemble_beta` is read ONLY from `RUNCONFIGS['flags-regression']` inside
`serverbase.init_ensemble_configs()`. It never looks at `args`, so putting it in
`OVERRIDES` does nothing. The authoritative value lives in
`utils/model_config.py`. This notebook forces it two ways:

1. **Preferred (permanent):** edit `utils/model_config.py` ->
   `RUNCONFIGS['flags-regression']['ensemble_beta'] = 1`. Then this notebook just
   confirms it.
2. **Runtime patch (no file edit):** the FIX cell below patches
   `RUNCONFIGS['flags-regression']` in memory BEFORE any server is built. Because
   the FedGen server calls `init_ensemble_configs()` in its constructor, it will
   read the patched value. This works for the current kernel session only.

### Hard gate
After α=0.01 trains, a verification cell parses the generator log for **nonzero
Student Loss**. If it's still 0.0, the sweep ABORTS with an explanation — so you
never burn a full sweep on a silently-disabled mechanism again.

### Still in force from 03b
- Restart the kernel before running (stale-import guard in the userbase cell).
- Best round / comparison on **scaled MAE** (unscaled is frozen by the
  `userbase.test()` inverse-scaling bug; MB values remain provisional).
- Partial arm uses **`FedGen-partial`** (genuine `mode='partial'`), not `pFedGen`.


In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import matplotlib.pyplot as plt
import pickle, json
import torch

print(f"Device: {uc2.DEFAULT_CONFIG['device']}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"| free {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

## Verify the fixed `userbase` is actually loaded (guards against stale import)

In [ ]:
import FLAlgorithms.users.userbase as ub
from FLAlgorithms.users.userpFedGen import UserpFedGen

ok = (hasattr(ub.User, "save_model")
      and hasattr(ub.User, "get_next_train_batch")
      and hasattr(UserpFedGen, "save_model")
      and hasattr(UserpFedGen, "get_next_train_batch"))
print("userbase loaded from:", ub.__file__)
print("User.save_model:", hasattr(ub.User, "save_model"),
      "| User.get_next_train_batch:", hasattr(ub.User, "get_next_train_batch"))
print("UserpFedGen.save_model:", hasattr(UserpFedGen, "save_model"),
      "| UserpFedGen.get_next_train_batch:", hasattr(UserpFedGen, "get_next_train_batch"))
assert ok, ("STALE IMPORT: restart the kernel (Kernel -> Restart) and run from the top. "
            "The fixed userbase.py is on disk but an old version is cached in memory.")
print("\n[OK] userbase is whole — safe to train.")

## Configuration

In [ ]:
ALPHAS = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
MODEL  = "lstm"

# This notebook targets the GLOBAL partition:
PARTITION_VARIANT = "client_local"      # -> new_partitions_global/ -> results/newpart_global/

RETRAIN = False                # first run on this variant: train, don't load stale pkls
RUN_PARTIAL = True             # also run the FedGen-partial arm

# --- distillation fix knobs (the whole point of 03c) ---
FORCE_ENSEMBLE_BETA = 1.0   # student-distillation weight; MUST be > 0 or FedGen==FedAvg
FORCE_ENSEMBLE_ALPHA = 1.0  # teacher weight (Zhu et al. default ~1)
FORCE_ENSEMBLE_ETA   = 1.0  # diversity weight (Zhu et al. default ~1)
ABORT_IF_STUDENT_LOSS_ZERO = True   # hard gate after the first alpha

OVERRIDES = dict(
    num_glob_iters=300,
    local_epochs=20,
    num_users=20,
    early_stopping_patience=25,
    ensemble_lr=1e-4,
    batch_size=32,
)

## Partition wiring (same mechanism as 01b / 02b, pinned to the global variant)

In [ ]:
VARIANT_SUBDIR  = {"client_local": "new_partitions",
                   "global":       "new_partitions_global"}
VARIANT_RESULTS = {"client_local": "newpart",
                   "global":       "newpart_global"}
assert PARTITION_VARIANT in VARIANT_SUBDIR, PARTITION_VARIANT

_NP_LOOKBACK = uc2.DEFAULT_CONFIG["lookback"]   # 60
_NP_STEPS    = uc2.DEFAULT_CONFIG["steps"]      # 1
_NP_SUBDIR   = VARIANT_SUBDIR[PARTITION_VARIANT]
_NP_RESULTS  = VARIANT_RESULTS[PARTITION_VARIANT]

_orig_make_args     = uc2.make_args
_orig_result_exists = uc2.result_exists
_orig_load_result   = uc2.load_result
_NEWPART_ON = {"flag": False}

def _newpart_dataset_path():
    return os.path.join(uc2.DATA_PART, _NP_SUBDIR,
                        f"lookback_{_NP_LOOKBACK}", f"steps_{_NP_STEPS}")

def _redirect_result_path(result_path, algorithm, alpha, model):
    base = os.path.join(uc2.RESULTS, _NP_RESULTS)
    if result_path is None:
        return os.path.join(base, algorithm.lower(), f"alpha_{alpha}", model, "rep_0")
    rel = os.path.relpath(os.path.abspath(result_path), os.path.abspath(uc2.RESULTS))
    return os.path.join(base, rel)

def _patched_make_args(algorithm, alpha, result_path=None, **overrides):
    args = _orig_make_args(algorithm, alpha, result_path=result_path, **overrides)
    if _NEWPART_ON["flag"]:
        args.dataset_path = _newpart_dataset_path()
        model = {**uc2.DEFAULT_CONFIG, **overrides}["model"]
        new_rp = _redirect_result_path(result_path, algorithm, alpha, model)
        new_rp = os.path.relpath(new_rp)
        os.makedirs(new_rp, exist_ok=True)
        args.result_path = new_rp
    return args

def _patched_result_exists(algorithm, alpha, model="lstm"):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, "rep_0", "full_results.pkl")
        return os.path.exists(p)
    return _orig_result_exists(algorithm, alpha, model)

def _patched_load_result(algorithm, alpha, model="lstm"):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, "rep_0", "full_results.pkl")
        with open(p, "rb") as f:
            return pickle.load(f)
    return _orig_load_result(algorithm, alpha, model)

def use_new_partitions(on=True):
    _NEWPART_ON["flag"] = bool(on)
    uc2.make_args     = _patched_make_args     if on else _orig_make_args
    uc2.result_exists = _patched_result_exists if on else _orig_result_exists
    uc2.load_result   = _patched_load_result   if on else _orig_load_result
    state = f"NEW [{PARTITION_VARIANT}]" if on else "ORIGINAL AP-ID"
    print(f"[wiring] partitions = {state}")
    if on:
        dp = _newpart_dataset_path()
        print(f"[wiring] dataset_path -> {dp}")
        print(f"[wiring] results      -> {os.path.join(uc2.RESULTS, _NP_RESULTS, '...')}")
        miss = [a for a in ALPHAS if not os.path.exists(
            os.path.join(dp, f"u{uc2.DEFAULT_CONFIG['n_users']}-alpha{a}-ratio1",
                         "train", "train.pt"))]
        if miss:
            print(f"[wiring] !! MISSING partitions for alpha={miss} -- run notebook 00 "
                  f"with test_mode='global' first!")

use_new_partitions(True)

In [ ]:
# --- VERIFY: client-local partition is wired and clients get distinct test sets ---
import torch
_args = uc2.make_args("FedGen", ALPHAS[0], **OVERRIDES)
print("dataset_path =", _args.dataset_path)
assert "new_partitions" in _args.dataset_path and "global" not in _args.dataset_path, \
    "WIRING OFF: dataset_path is not the client-local new_partitions root."

_server = uc2.create_server(_args)
print(f"\n{'client':>8}{'test n':>10}{'target mean':>14}")
means = []
for c in _server.users[:5]:
    xb, yb = next(iter(c.testloaderfull))   # full test set for this client
    yy = yb.float()
    means.append(round(float(yy.mean()), 4))
    print(f"{str(c.id):>8}{c.test_samples:>10}{yy.mean():>14.4f}")

assert len(set(means)) > 1, "Clients have identical test means -> still shared test pool."
print("\n[OK] Clients have DISTINCT test sets. Client-local wiring confirmed. "
      "Safe to run the sweep below.")

## FIX — force `ensemble_beta > 0` (the distillation weight)

Patches `RUNCONFIGS['flags-regression']` in memory so every FedGen server built
afterwards reads the corrected weights. Prints before/after so you can see what
changed. If you've already edited `utils/model_config.py`, this is idempotent
(it just re-asserts the value).

In [ ]:
from utils.model_config import RUNCONFIGS
from utils.model_utils import get_dataset_name

# resolve the dataset key the server will use (e.g. "flags-regression")
_probe_args = uc2.make_args("FedGen", ALPHAS[0], **OVERRIDES)
DS_KEY = get_dataset_name(_probe_args.dataset)
print("dataset key:", DS_KEY)

cfg = RUNCONFIGS.setdefault(DS_KEY, {})
before = {k: cfg.get(k, "(missing -> default)")
          for k in ["ensemble_alpha", "ensemble_beta", "ensemble_eta",
                    "generative_alpha", "generative_beta"]}
print("\nBEFORE:", before)

# apply the fix
cfg["ensemble_beta"]  = FORCE_ENSEMBLE_BETA
cfg["ensemble_alpha"] = FORCE_ENSEMBLE_ALPHA
cfg["ensemble_eta"]   = FORCE_ENSEMBLE_ETA

after = {k: RUNCONFIGS[DS_KEY].get(k) for k in
         ["ensemble_alpha", "ensemble_beta", "ensemble_eta",
          "generative_alpha", "generative_beta"]}
print("AFTER: ", after)

assert RUNCONFIGS[DS_KEY]["ensemble_beta"] > 0, \
    "ensemble_beta is still 0 — the patch did not take."
print(f"\n[FIX] ensemble_beta = {RUNCONFIGS[DS_KEY]['ensemble_beta']}  "
      f"(>0 -> student distillation is now ACTIVE)")
print("[FIX] NOTE: this is an in-memory patch for THIS kernel only. For a")
print("      permanent fix, set ensemble_beta in utils/model_config.py.")

## Helpers — select & report on SCALED MAE

`best_scaled` is the source of truth for "best round". `best_pair` returns
(scaled, unscaled) at the scaled-best round so you can see both, but rankings and
plots use scaled.

In [ ]:
def best_round_idx(r):
    glob = r["metrics"].get("glob_test_metric", [])
    if not glob:
        return None
    # select on SCALED mae (unscaled is unreliable for FedGen on this data)
    return int(np.argmin([m.get("mae", float("inf")) for m in glob]))

def best_scaled(r):
    i = best_round_idx(r)
    if i is None: return float("nan")
    return r["metrics"]["glob_test_metric"][i].get("mae", float("nan"))

def best_pair(r):
    i = best_round_idx(r)
    if i is None: return float("nan"), float("nan")
    m = r["metrics"]["glob_test_metric"][i]
    return m.get("mae", float("nan")), m.get("unscaled_mae", float("nan"))

def report(tag, alpha, r):
    i = best_round_idx(r)
    if i is None:
        print(f"  [{tag}] α={alpha}: no glob metrics"); return
    sc, un = best_pair(r)
    print(f"  [{tag}] α={alpha}: best round {i} | scaled MAE={sc:.4f} "
          f"| unscaled MAE={un:.4f} (provisional)")
    pu = r.get("per_user", {}).get("metrics", {}).get("unscaled_mae", [])
    if pu:
        print(f"           per-user unscaled MAE: mean={np.mean(pu):.3f} "
              f"std={np.std(pu):.3f} CV={np.std(pu)/np.mean(pu):.3f}")

In [ ]:
# Re-patch the new-partition wiring to thread `seed` through (CELL 7 hardcoded rep_0).
def _redirect_result_path(result_path, algorithm, alpha, model, seed=0):
    base = os.path.join(uc2.RESULTS, _NP_RESULTS)
    if result_path is None:
        return os.path.join(base, algorithm.lower(), f"alpha_{alpha}", model, f"rep_{seed}")
    rel = os.path.relpath(os.path.abspath(result_path), os.path.abspath(uc2.RESULTS))
    return os.path.join(base, rel)

def _patched_make_args(algorithm, alpha, result_path=None, seed=0, **overrides):
    args = _orig_make_args(algorithm, alpha, result_path=result_path, seed=seed, **overrides)
    if _NEWPART_ON["flag"]:
        args.dataset_path = _newpart_dataset_path()
        model = {**uc2.DEFAULT_CONFIG, **overrides}["model"]
        new_rp = os.path.relpath(_redirect_result_path(result_path, algorithm, alpha, model, seed=seed))
        os.makedirs(new_rp, exist_ok=True)
        args.result_path = new_rp
    return args

def _patched_result_exists(algorithm, alpha, model="lstm", seed=0):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, f"rep_{seed}", "full_results.pkl")
        return os.path.exists(p)
    return _orig_result_exists(algorithm, alpha, model, seed=seed)

def _patched_load_result(algorithm, alpha, model="lstm", seed=0):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, f"rep_{seed}", "full_results.pkl")
        with open(p, "rb") as f:
            return pickle.load(f)
    return _orig_load_result(algorithm, alpha, model, seed=seed)

uc2.make_args     = _patched_make_args
uc2.result_exists = _patched_result_exists
uc2.load_result   = _patched_load_result
print("[wiring] seed-aware patch active. result_exists takes seed:",
      "seed" in __import__("inspect").signature(uc2.result_exists).parameters)

## Part A — FedGen (full) on the global partition

Algorithm string `"FedGen"`. Full model + generator exchange.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Run FedGen seeds. seed 0 is already on disk under rep_0 and is NOT in this list.
SEEDS = [42, 123]

results_fedgen_seeds = {}        # keyed (alpha, seed)
for seed in SEEDS:
    for alpha in ALPHAS:
        print(f"\n{'='*60}\n  FedGen (full) [{PARTITION_VARIANT}] — α={alpha} seed={seed}\n{'='*60}")
        if not RETRAIN and uc2.result_exists("FedGen", alpha, seed=seed):
            print("  [done] loading cached result.")
            results_fedgen_seeds[(alpha, seed)] = uc2.load_result("FedGen", alpha, seed=seed)
            continue
        try:
            server, result = uc2.run_experiment(
                algorithm="FedGen", alpha=alpha, seed=seed, **OVERRIDES)
            results_fedgen_seeds[(alpha, seed)] = result
            report("full", alpha, result)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  [ERROR] {e}")

## Part B — FedGen (partial) on the global partition

Algorithm string **`"FedGen-partial"`** — contains `"partial"`, so `serverbase`
sets `mode='partial'` and only the shared decoder layer is aggregated. This is the
*genuine* low-communication variant (unlike `pFedGen`, which exchanges the full
model). Results go to `results/newpart_global/fedgen-partial/...`.

In [ ]:
results_fedgen_seeds_partial = {}

# Run FedGen seeds. seed 0 is already on disk under rep_0 and is NOT in this list.
SEEDS = [42, 123]
       # keyed (alpha, seed)
for seed in SEEDS:
    for alpha in ALPHAS:
        print(f"\n{'='*60}\n  FedGen (partial) [{PARTITION_VARIANT}] — α={alpha} seed={seed}\n{'='*60}")
        if not RETRAIN and uc2.result_exists("FedGen-Partial", alpha, seed=seed):
            print("  [done] loading cached result.")
            results_fedgen_seeds_partial[(alpha, seed)] = uc2.load_result("FedGen-Partial", alpha, seed=seed)
            continue
        try:
            server, result = uc2.run_experiment(
                algorithm="FedGen-Partial", alpha=alpha, seed=seed, **OVERRIDES)
            results_fedgen_seeds_partial[(alpha, seed)] = result
            report("partial", alpha, result)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  [ERROR] {e}")

## Convergence — full vs partial (scaled MAE)

In [ ]:
uc2.setup_plot_style()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax = axes[0]
for alpha, r in sorted(results_fedgen_seeds.items()):
    sc = [m.get("mae") for m in r["metrics"].get("glob_test_metric", [])]
    if sc: ax.plot(sc, label=f"α={alpha}", lw=2)
ax.set_xlabel("Communication round"); ax.set_ylabel("Scaled MAE")
ax.set_title("FedGen (full) — global"); ax.legend()

ax = axes[1]
for alpha, r in sorted(results_fedgen_seeds_partial.items()):
    sc = [m.get("mae") for m in r["metrics"].get("glob_test_metric", [])]
    if sc: ax.plot(sc, label=f"α={alpha}", lw=2)
ax.set_xlabel("Communication round")
ax.set_title("FedGen (partial) — global")
if results_fedgen_seeds_partial: ax.legend()

plt.suptitle("FedGen convergence (scaled MAE) — global partition", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
uc2.setup_plot_style()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax = axes[0]
for alpha, r in sorted(results_fedgen_seeds.items()):
    sc = [m.get("unscaled_mae") for m in r["metrics"].get("glob_test_metric", [])]
    if sc: ax.plot(sc, label=f"α={alpha}", lw=2)
ax.set_ylim(0, 100)     
ax.set_xlabel("Communication round"); ax.set_ylabel("Unscaled MAE")
ax.set_title("FedGen (full) — global"); ax.legend()

ax = axes[1]
for alpha, r in sorted(results_fedgen_seeds_partial.items()):
    sc = [m.get("unscaled_mae") for m in r["metrics"].get("glob_test_metric", [])]
    if sc: ax.plot(sc, label=f"α={alpha}", lw=2)
ax.set_ylim(0, 100)     
ax.set_xlabel("Communication round")
ax.set_title("FedGen (partial) — global")
if results_fedgen_seeds_partial: ax.legend()

plt.suptitle("FedGen convergence (unscaled MAE) — global partition", fontsize=14)
plt.tight_layout(); plt.show()

## Best-MAE summary (scaled primary, unscaled provisional)

In [ ]:
def dump(tag, res):
    if not res:
        print(f"[{tag}] no results"); return
    print(f"\n[{tag}]")
    print(f"{'α':<8}{'scaled MAE':>12}{'unscaled MAE':>14}{'best round':>12}{'rounds':>8}")
    for a in sorted(res):
        r = res[a]; i = best_round_idx(r); sc, un = best_pair(r)
        print(f"{a:<8}{sc:>12.4f}{un:>14.4f}{(i if i is not None else -1):>12}"
              f"{r.get('n_rounds', 0):>8}")

dump("FedGen full", results_fedgen_seeds)
dump("FedGen partial", results_fedgen_seeds_partial)

## Method comparison on the global pool: Centralized vs FedAvg vs FedGen

All four are read from `results/newpart_global/...`, i.e. the SAME global test
pool, so the comparison is apples-to-apples. Requires `01b` (Centralized) and
`02b` (FedAvg) to have been run with `PARTITION_VARIANT="global"`. Missing methods
are skipped. Comparison is on **scaled MAE**.

In [ ]:
def _load_global(algorithm, alpha):
    p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                     f"alpha_{alpha}", MODEL, "rep_0", "full_results.pkl")
    if not os.path.exists(p): return None
    with open(p, "rb") as f: return pickle.load(f)

series = {}
series["Centralized"]    = {a: _load_global("Centralized", a) for a in ALPHAS}
series["FedAvg"]         = {a: _load_global("FedAvg", a)      for a in ALPHAS}
series["FedGen"]         = {a: (results_fedgen_seeds.get(a)    or _load_global("FedGen", a))         for a in ALPHAS}
series["FedGen-partial"] = {a: (results_fedgen_seeds_partial.get(a) or _load_global("FedGen-partial", a)) for a in ALPHAS}

style = {"Centralized": ("s--", uc2.COLORS.get("Centralized", "k")),
         "FedAvg":      ("o-",  uc2.COLORS.get("FedAvg", "C0")),
         "FedGen":      ("^-",  uc2.COLORS.get("FedGen", "C1")),
         "FedGen-partial": ("v-", uc2.COLORS.get("FedGen", "C2"))}

fig, ax = plt.subplots(figsize=(10, 5))
have_any = False
for name, res in series.items():
    pts = [(a, best_scaled(r)) for a, r in res.items() if r is not None]
    if not pts: continue
    have_any = True
    xs, ys = zip(*sorted(pts))
    fmt, col = style[name]
    ax.plot(xs, ys, fmt, color=col, lw=2, label=name)
ax.set_xscale("log"); ax.set_xlabel("α (log)"); ax.set_ylabel("Best scaled MAE")
ax.set_title("Global pool: methods vs α (scaled MAE)")
if have_any: ax.legend()
plt.tight_layout(); plt.show()

# gap-to-Centralized table (scaled)
cent = series["Centralized"]
if any(v is not None for v in cent.values()):
    print(f"{'α':<8}{'Central':>10}{'FedAvg':>10}{'FedGen':>10}{'FGpart':>10}")
    for a in sorted(ALPHAS):
        def g(name):
            r = series[name].get(a); return best_scaled(r) if r is not None else float('nan')
        print(f"{a:<8}{g('Centralized'):>10.4f}{g('FedAvg'):>10.4f}"
              f"{g('FedGen'):>10.4f}{g('FedGen-partial'):>10.4f}")
else:
    print("[info] No global Centralized results — run 01b with PARTITION_VARIANT='global'.")

## Communication cost — and a correctness note

The previous notebook reported a flat **98.4% reduction** for "partial". That came
from a FORMULA keyed to the label, and the run it described used `pFedGen`
(`mode='all'`), which actually exchanged the FULL model — so the figure was
nominal, not real. This notebook runs `"FedGen-partial"` (`mode='partial'`), so the
partial arm genuinely exchanges only the shared layer and the cost below reflects
what was transmitted. The per-α numbers also now vary because they use the actual
`n_rounds` each run reached.

In [ ]:
print(f"{'α':>6} | {'Full (MB)':>12} | {'Partial (MB)':>14} | {'Reduction':>10}")
print("-"*52)
for a in ALPHAS:
    rf = results_fedgen_seeds.get(a); rp = results_fedgen_seeds_partial.get(a)
    n_full = rf.get("n_rounds", OVERRIDES["num_glob_iters"]) if rf else None
    n_part = rp.get("n_rounds", OVERRIDES["num_glob_iters"]) if rp else None
    if n_full is None:
        print(f"{a:>6} | {'--':>12} | {'--':>14} | {'--':>10}"); continue
    c_full = uc2.comm_cost_fedgen_full(n_full, OVERRIDES["num_users"])
    if n_part is None:
        print(f"{a:>6} | {c_full:>12.1f} | {'--':>14} | {'--':>10}"); continue
    c_part = uc2.comm_cost_fedgen(n_part, OVERRIDES["num_users"])
    red = (1 - c_part / c_full) * 100 if c_full > 0 else 0
    print(f"{a:>6} | {c_full:>12.1f} | {c_part:>14.1f} | {red:>9.1f}%")

## Notes & known issues carried forward

- **Metric:** scaled MAE is primary; unscaled MB is provisional until the
  inverse-scaling in `userbase.test()` is verified (it froze `unscaled_mae` for
  FedGen in the prior run). MAPE is meaningless here (near-zero load → ~1e9+).
- **Partial means partial now:** `"FedGen-partial"` engages `mode='partial'`.
  Do not revert to `pFedGen` for the partial arm.
- **FedGen-specific things to watch as α/seeds change:** the generator GMM is fit
  with `n_components = n_selected_users = 20`, which is seed-fragile on this
  smooth, mild-label-skew data; and the y-conditioning dimension differs between
  the plain and "Specified" FedGen server variants. If FedGen training is
  unstable, these are the first suspects.
- **For the client-local view:** set `PARTITION_VARIANT="client_local"` and
  re-run; results land under `results/newpart/...` next to your FedAvg client-local
  run.


In [ ]:
import pickle, os, numpy as np
for a in [0.01, 0.1, 10.0]:
    r = pickle.load(open(os.path.join(uc2.RESULTS,"newpart","fedgen",
                         f"alpha_{a}","lstm","rep_0","full_results.pkl"),"rb"))
    pm = np.array(r["per_user"]["metrics"]["mae"])
    ns = np.array(r["per_user"]["num_samples"])
    order = np.argsort(ns)
    print(f"\nα={a} (FedGen):")
    print(f"  unweighted per-client MAE: mean={pm.mean():.4f}  median={np.median(pm):.4f}")
    print(f"  sample-weighted MAE       : {np.sum(pm*ns)/np.sum(ns):.4f}")
    print(f"  smallest-5 clients: {pm[order[:5]].mean():.4f}   largest-5: {pm[order[-5:]].mean():.4f}")